# CLIP Clustering

This notebook walks you through the code that was experimented with to cluster the Glashelder dataset using CLIP embeddings.

The main technologies being used are:
- [CLIP pretrained model](https://huggingface.co/laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K)
- [UMAP dimensionality reduction algorithm](https://umap-learn.readthedocs.io/en/latest/)
- [HDBSCAN density based spatial clustering algorithm for applications with noise](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)

**CLIP** is a machine learning model by OpenAI trained to encode both images and text into a shared latent space as high-dimensional vectors called embeddings. The space has semantic structure: images with similar subjects end up close together, while dissimilar images are further apart.

**UMAP** is a non-linear dimensionality reduction algorithm. CLIP embeddings have 768 dimensions; too high for clustering algorithms to work efficiently (the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality)). UMAP reduces the data while preserving its underlying structure as well as possible. We reduce to 30 dimensions for clustering, and separately to 3 dimensions for visualization.

**HDBSCAN** is a density-based clustering algorithm that groups points into clusters based on how densely packed they are in the embedding space. Unlike many clustering algorithms, it supports a noise label (`-1`) for points that don't clearly belong to any cluster; which is desirable here, since not every image is expected to fit neatly into a category.

The data flow of the code in this notebook looks like this:

```mermaid
flowchart LR
  i[images]@{shape: procs} -->|N images| c[CLIP]
  c -->|embedding 768xN| u1["UMAP(1)"]
  u1 -->|reduced 30xN| h
  u1 -->|reduced 30xN| u2["UMAP(2)"]
  u2 -->|reduced 3xN| v[Rerun Visualization]
  h[HDBSCAN] --> clusters@{shape: procs}
  h --> noise
  noise --> v
  clusters --> v
```




# Setup

### Dependencies

Import all necessary python libraries by running the cell below:

In [ ]:
import glob  # for going through files in a directory

from hdbscan import (
    HDBSCAN,
)  # hierarchical density based spatial clustering of applications with noise
from umap import UMAP  # nonlinear dimensionality reduction

from cluster.rr import (
    log_clustered_points,
    log_images_3d,
)  # logging points and images to the rerun visualization
from common import PROJECT_ROOT  # top level path of this project
from embed import (
    CLIPEmbedder,
    get_device,
)  # getting clipmodel and device (gpu, cpu) where it runs

### Configurations
- `MODEL`: the CLIP model to get from huggingface
- `PACKAGE_ROOT`: filesystem path of the directory with files related to the `cluster` experiment
- `EMBEDDINGS_PATH`: the location where the embeddings will be saved to in a file
- `IMAGE_DIR`: the path where you store your Glashelder images. **Modify the path below to where your Glashelder images are located**.

In [ ]:
MODEL = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"
PACKAGE_ROOT = PROJECT_ROOT / "cluster"
EMBEDDINGS_PATH = PACKAGE_ROOT / "embeddings.pkl"
IMAGE_DIR = "/mnt/share/shares/glashelder/assets-crop"  # MODIFY ME

Download the model and set it up for your device (cuda, cpu, mps):

In [ ]:
device = get_device()
embedder = CLIPEmbedder(MODEL)
embedder.load(device)

## Make image embeddings

A random selection of images is chosen from within the image directory (defined above in `ÌMAGE_DIR`) that was specified before. The amount of images is limited here to a dataset of 4000 for speed. If you want a larger dataset to look through, increase the variable, `count`.

In [ ]:
# this makes a 'random' (images have random order) selection of images in the dataset of limited size for speed

count = 4000  # If you want to select a larger portion of your dataset to look through, increase this variable
image_paths = sorted(glob.glob(f"{IMAGE_DIR}/*.jpg"))[:count]

Generate the embeddings of the images:

> `batch_size` controls how many images are sent to the GPU at once to compute in parallel. \
> If you encounter Out of Memory errors, you should reduce `batch_size` to something lower (e.g. `32`, `8`, even `1`): \
> this will require less VRAM, but take longer to compute.

In [ ]:
embeddings, image_paths = embedder.encode_images(image_paths, batch_size=64)
embeddings.shape  # count vectors of length 768

# Reduce embeddings' dimensionality

In the previous block of code we generated embeddings with 768 dimensions. This is too high for clustering algorithms to work efficiently. We reduce to 30 dimensions below:

In [ ]:
# 30 dimensions for clustering
umap_model_30 = UMAP(n_components=30, n_neighbors=15, min_dist=0.20)
embeddings_30 = umap_model_30.fit_transform(embeddings)

30 dimensions is efficient for clustering but impossible to visualize. A second UMAP reduction brings the data down to 3 dimensions so the clusters can be explored in a 3D viewer:

In [ ]:
# 3 dimensions for visualization
umap_model_3 = UMAP(n_components=3, n_neighbors=15, min_dist=0.20)
embeddings_3 = umap_model_3.fit_transform(embeddings_30)

# Cluster in 30 dimensions

HDBSCAN is fit on the 30-dimensional embeddings. Each image is assigned a cluster label — images that don't clearly belong to any cluster get label `-1` (noise). Inspect `clusterer.labels_` to see the label assigned to each image.

In [ ]:
clusterer = HDBSCAN(min_cluster_size=10)
clusterer.fit(embeddings_30)

In [ ]:
clusterer.labels_

# Show clusters
Chances are high that the clustering doesn't return desired results when you supply it a limited number of images. You can mitigate this by increasing the `count` variable at the top half of this notebook.

Import the necessary python libraries by running the cell below:

In [ ]:
import rerun as rr

Log the 3D embeddings and a random sample of images to a [Rerun](https://rerun.io/) recording file. Points are colored by cluster label; noise points are shown separately. Open `clusters.rrd` in the Rerun viewer to explore the result interactively.

In [ ]:
rr.init("cluster_notebook")
rr.save("clusters.rrd")
log_clustered_points(embeddings_3, clusterer.labels_, count=count)
log_images_3d(embeddings_3, image_paths, 100)

Open the saved file 'clusters.rrd' in the rerun viewer (e.g. by first downloading it from the jupyter UI)